**Part 3 · LangGraph** · step 20 · Human-in-the-loop graph  
[← 19 Research chatbot](19_research_chatbot_multiple_tools.ipynb) · [🏠 Roadmap](../README.md) · [Next: When to use what →](../04_When_To_Use_What/20_choosing_the_right_tool.md)

# 20 · Human-in-the-Loop in a Graph (`interrupt` + `Command(resume=...)`)

Sometimes the agent should **stop and ask a person** before it continues, for example when it needs expert
advice, an approval, or a missing detail. LangGraph does this with `interrupt()`.

The idea in one picture:
```
graph.invoke(question)          ──► chatbot ──► tools ──► human_assistance() hits interrupt()  ⏸ PAUSE
                                                                     │  state saved by checkpointer
graph.invoke(Command(resume=…)) ─────────────────────────────────────┘  interrupt() returns the human's answer
                                                                        ──► chatbot ──► END
```

| Piece | Job |
|---|---|
| `interrupt(value)` | Pauses the graph. `value` is what the human sees (e.g. the question). |
| checkpointer (`InMemorySaver`) | Saves the paused state. **Without it you can't resume.** |
| `thread_id` in `config` | Says *which* paused conversation to continue. Use the **same one** to resume. |
| `Command(resume=answer)` | Continues the run. `answer` becomes the return value of `interrupt()`. |

In notebook 14 you used `HumanInTheLoopMiddleware` with `create_agent`. That middleware is built on this same `interrupt()`.

In [ ]:
from dotenv import load_dotenv

# Reads API keys (OPENAI_API_KEY, GROQ_API_KEY, GOOGLE_API_KEY, TAVILY_API_KEY)
# from the .env file at the AgenticAI root. load_dotenv() walks up parent
# folders until it finds one, so every notebook can share a single .env.
load_dotenv()

In [ ]:
from typing import Annotated
from typing_extensions import TypedDict

from langchain.chat_models import init_chat_model
from langchain.messages import AnyMessage, HumanMessage
from langchain.tools import tool
from langchain_tavily import TavilySearch
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import StateGraph, START
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition
from langgraph.types import Command, interrupt

## 1. A tool that asks a human
`human_assistance` looks like a normal tool to the LLM. Inside, `interrupt()` pauses the whole graph.
When you resume, `interrupt()` returns whatever you passed in `Command(resume=...)`, and the tool returns it
to the LLM as its result.

In [ ]:
@tool
def human_assistance(query: str) -> str:
    """Request assistance from a human expert."""
    answer = interrupt({"query": query})  # ⏸ pauses here until Command(resume=...)
    return answer["data"]                 # so resume with a dict that has a "data" key

## 2. Tools + LLM
The LLM can choose between web search (Tavily) and asking a human.

In [ ]:
tavily = TavilySearch(max_results=2)

tools = [human_assistance, tavily]
llm_with_tools = init_chat_model("gpt-4o").bind_tools(tools)

## 3. The graph
Same ReAct loop as notebook 18. The only new part is `compile(checkpointer=...)`, because a paused graph has to be saved somewhere.

⚠️ The state key **must be `messages`** (plural). `tools_condition` and `ToolNode` look for `state["messages"]`.

In [ ]:
class State(TypedDict):
    """Chat history; add_messages appends new messages."""

    messages: Annotated[list[AnyMessage], add_messages]


def chatbot(state: State) -> dict:
    """Call the LLM: answer directly or ask for a tool (search or human)."""
    return {"messages": [llm_with_tools.invoke(state["messages"])]}


builder = StateGraph(State)
builder.add_node("chatbot", chatbot)
builder.add_node("tools", ToolNode(tools))

builder.add_edge(START, "chatbot")
builder.add_conditional_edges("chatbot", tools_condition)  # -> "tools" or END
builder.add_edge("tools", "chatbot")

graph = builder.compile(checkpointer=InMemorySaver())  # checkpointer = required for interrupt + resume

In [ ]:
from IPython.display import Image, display


def show_graph(compiled_graph):
    """Render a compiled graph as a PNG.

    draw_mermaid_png() uses the mermaid.ink web service; if you are offline
    it falls back to printing the Mermaid source (paste it into mermaid.live).
    """
    try:
        display(Image(compiled_graph.get_graph().draw_mermaid_png()))
    except Exception:
        print(compiled_graph.get_graph().draw_mermaid())


show_graph(graph)

## 4. Run until it pauses
The run stops inside `human_assistance`. Instead of an error you get a normal result with an extra
`__interrupt__` key that holds what the graph is asking.

In [ ]:
config = {"configurable": {"thread_id": "1"}}

resp = graph.invoke(
    {"messages": [HumanMessage("I need some expert guidance for building an AI agent. Could you request assistance for me?")]},
    config,
)

for m in resp["messages"]:
    m.pretty_print()

print("\n⏸ Graph is asking:", resp["__interrupt__"][0].value)

### Peek at the paused state
`get_state(config)` shows where the graph stopped. `next` is the node waiting to finish.

In [ ]:
snapshot = graph.get_state(config)
print("waiting on node:", snapshot.next)                          # ('tools',)
print("pending tool call:", snapshot.values["messages"][-1].tool_calls)

## 5. Resume with the human's answer
Same `config` (same `thread_id`), and a `Command(resume=...)` **instead of** new input.
The dict shape must match what the tool reads: `answer["data"]`.

(In a real app this text would come from a form or chat UI. In a notebook you could also use `input()`.)

In [ ]:
human_response = (
    "We, the experts, are here to help! We'd recommend you check out LangGraph to build your agent."
    " It's much more reliable and extensible than simple autonomous agents."
)

resp = graph.invoke(Command(resume={"data": human_response}), config)

for m in resp["messages"][-2:]:  # the ToolMessage (human's answer) + the LLM's final reply
    m.pretty_print()

print("\nwaiting on node:", graph.get_state(config).next)  # () = finished

## 6. The same flow with streaming
With `stream_mode="updates"`, the pause shows up as a chunk with an `__interrupt__` key (streaming is covered in notebook 18).
A new `thread_id` gives you a fresh conversation.

In [ ]:
config2 = {"configurable": {"thread_id": "2"}}
question = {"messages": [HumanMessage("Ask a human expert: should I use LangGraph or create_agent for a simple FAQ bot?")]}

for chunk in graph.stream(question, config2, stream_mode="updates"):
    if "__interrupt__" in chunk:
        print("⏸ paused, human is asked:", chunk["__interrupt__"][0].value["query"])
    else:
        print("step:", list(chunk))

In [ ]:
answer = {"data": "For a simple FAQ bot, create_agent is enough. Use LangGraph when you need custom flow."}

for chunk in graph.stream(Command(resume=answer), config2, stream_mode="updates"):
    for node, update in chunk.items():
        print(f"step: {node}")

print("\nFinal answer:", graph.get_state(config2).values["messages"][-1].content)

## ⚠️ Mistakes made (and fixed) while learning this
| Mistake | What you see | Fix |
|---|---|---|
| State key `message` instead of `messages` | `ValueError: No messages found in input state to tool_edge` | Name it `messages`. `tools_condition` / `ToolNode` read that key |
| Fixed the class but still invoked with `{"message": ...}` | OpenAI `400: Invalid 'messages': empty array` | LangGraph **silently ignores** unknown input keys, so the LLM got 0 messages. Use the same key everywhere |
| Resumed with a plain string: `Command(resume="...")` | `TypeError: string indices must be integers` | The tool reads `answer["data"]`, so resume with `{"data": "..."}` |
| Resumed with a **different `thread_id`** | A brand-new run with no messages (again the empty-array error) | Resume with the exact same `config` |
| No checkpointer | It pauses, but there's nothing saved to resume | `compile(checkpointer=InMemorySaver())` |
| Re-running cells out of order | Old `State` / `graph` still in memory, so a fix "doesn't work" | Restart the kernel and run all |

**Good to know:** when you resume, the node that paused **runs again from the top**, and `interrupt()` returns
the answer the second time. So don't put side effects (sending an email, writing to a DB) *before* `interrupt()` in the same node.

## Revision notes (quick read)
- **What:** `interrupt()` = "pause and ask a human". `Command(resume=...)` = "here's the answer, carry on".
- **Needs two things:** a **checkpointer** (to save the pause) and the **same `thread_id`** (to find it again).
- **Flow:**
  1. `graph.invoke(input, config)` → result has `__interrupt__` (what's being asked).
  2. Show that to a human, get their answer.
  3. `graph.invoke(Command(resume=answer), config)` → `interrupt()` returns `answer`, the graph continues.
- **Shapes must match:** whatever you pass to `resume` is exactly what `interrupt()` returns.
- **Check where it stopped:** `graph.get_state(config).next`. `('tools',)` = paused, `()` = done.
- **Streaming:** the pause arrives as a `{"__interrupt__": ...}` chunk.
- **Keys:** use `messages` everywhere (state, node return, input). Wrong input keys are ignored silently.
- **Resume re-runs the paused node from the top.** Keep side effects after `interrupt()`.
- **Shortcut:** with `create_agent`, `HumanInTheLoopMiddleware` (notebook 14) gives approve / edit / reject on tool calls without writing this yourself.

---
✅ **Done with 20 Human-in-the-loop graph.** Next: [When to use what →](../04_When_To_Use_What/20_choosing_the_right_tool.md)  
[🏠 Roadmap](../README.md)